# LTX-2.5 — Colab生成ノートブック（窓際族物語 / colab-video スキル・製品生成専用）

動画スキルで制作済みのバンドル（キーフレーム＋`ch*_workflow.json`）を、ColabのGPU（L4/A100）でチャプター毎に動画化する。
`h3_colab.ipynb`のLTX-2.5版。**セリフチャプターは対象外**（LTX-2.5には添付wav駆動のリップシンクが無い）— セリフはH3ノートブックで生成し、こちらは**セリフなし（I2V）チャプター専用**。音声（環境音）はプロンプトのSoundscape/Music記述から自動生成される。

H3との違い: 重みはint8一式・約39GBがL4/A100共通（GPU別バリアント切替なし）で、全ファイルを常時ローカルに置けるためユニット入れ替えも無い。フレーム数は**8k+1グリッド**（97, 121, 145, 193, ...）@24fps（H3の17k+5とは違う）。蒸留8ステップ生成。

初回のみ: https://huggingface.co/Lightricks/LTX-2.5 で「Agree and Access」を押してライセンス承諾し、READ権限のHFトークンをColabのシークレット（左の鍵アイコン → 名前 `HF_TOKEN`・ノートブックからのアクセスON）に登録する（**ゲート付きリポジトリのため重みDL時に必須**。Drive配置済みなら不要）。

使い方: **セル1だけ編集**したら、その直下の**★一括実行セル**を押す（セル1〜7を続けて流す＝1つずつ押す必要はない）。途中で失敗したら原因を直し、そのセルの`FROM_STEP`を失敗した番号にして再実行すれば続きから流せる。1つずつ確認しながら進めたいときは従来どおりセル1→8を順に実行してもよい。パイロットを1本生成して確認してから残りを回すこと。

課金の注意: CUは「GPUランタイム接続中の時間」で消費される（セル実行中でなくても）。終わったら必ず「ランタイム → ランタイムを接続解除して削除」。セル1の`AUTO_SHUTDOWN = True`にすると、セル7の全チャプター完了後に自動で削除される（Drive退避運用時のみ）。


In [ ]:
#@title 0.（初回のみ・無料CPUランタイムでOK）重みをDriveへ事前配置 — GPUセッションのCU消費とDL待ちをなくす
# 使い方: 「ランタイム → ランタイムのタイプを変更 → CPU」にして、このセルだけを実行する（セル1以降は不要）。
# 事前にHFのライセンス承諾とHF_TOKENシークレット登録（ヘッダの説明）を済ませておくこと。
# Driveに完全な重みが揃っていれば何もしないので、再実行は常に安全。完了後はこのCPUランタイムを削除してよい。
DRIVE_DIR = "/content/drive/MyDrive/ltx25_weights"  # GPUセッションのWEIGHTS_DRIVE_DIRと同じ場所
INCLUDE_UPSCALERS = False  # 二段構成（潜在アップスケーラー）用。既定の単段生成では不要

import os, shutil, subprocess, urllib.error, urllib.request
REPO = "https://huggingface.co/Lightricks/LTX-2.5/resolve/main"
SIZES = {  # HF上の正確なバイト数（2026-08時点）。完全性チェックに使う
    "diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors": 21_504_034_224,
    "text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors": 15_372_971_786,
    "vae/ltx-2.5-video-vae-bf16.safetensors": 1_472_223_346,
    "vae/ltx-2.5-audio-vae-bf16.safetensors": 364_866_540,
    "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors": 995_778_752,
    "latent_upscale_models/ltx-2.5-latent-temporal-upscaler-x2-bf16-1.0.safetensors": 261_944_000,
}
need = [r for r in SIZES if INCLUDE_UPSCALERS or "upscaler" not in r]

def hf_token():
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            return t
    except Exception:
        pass
    import getpass
    return getpass.getpass("HFトークン（read・シークレット未登録のため手入力）: ")

def resolved_url(rel, token):
    # ゲート付きリポジトリ: トークン付きでリダイレクトを解決し、署名済みCDN URLを得てから
    # aria2はヘッダ無しでDLする（AuthorizationヘッダをCDNへ転送すると拒否されることがあるため）。
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, *a, **k):
            return None
    req = urllib.request.Request(f"{REPO}/{rel}", method="HEAD",
                                 headers={"Authorization": f"Bearer {token}"})
    try:
        urllib.request.build_opener(NoRedirect).open(req, timeout=60)
        return req.full_url  # リダイレクトなし＝そのままDL可
    except urllib.error.HTTPError as e:
        if e.code in (301, 302, 303, 307, 308):
            return e.headers["Location"]
        if e.code in (401, 403):
            raise SystemExit(
                "HFのライセンス未承諾かトークン権限不足 — https://huggingface.co/Lightricks/LTX-2.5 "
                "で Agree and Access を押し、READ権限（gated reposアクセス可）のトークンを使う")
        raise

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
if not shutil.which("aria2c"):
    subprocess.run(["apt-get", "-qq", "-y", "install", "aria2"], check=True, capture_output=True)

todo = []
for rel in need:
    n = os.path.basename(rel)
    dst = f"{DRIVE_DIR}/{n}"
    if os.path.exists(dst) and os.path.getsize(dst) == SIZES[rel]:
        print("OK（配置済み）", n)
    else:
        todo.append(rel)
total = sum(SIZES[r] for r in todo)
free = shutil.disk_usage(DRIVE_DIR).free
print(f"これから配置: {len(todo)}本 / {total/2**30:.1f} GiB（Drive空き {free/2**30:.1f} GiB）")
assert free >= total + 2_000_000_000, (
    "Drive空き不足。h3_weights（約75GB）と併存できるかはSKILL.mdのLTX-2.5章の容量指針を参照"
    "（H3を片ユニット運用にして空ける / 200GBプランにする / LTXはDrive配置せず毎セッションDLする）。"
    "削除はゴミ箱行きで容量にカウントされ続けるので、削除後はゴミ箱も空にすること")

TOKEN = hf_token() if todo else None
TMP = "/content/ltx_dl"
os.makedirs(TMP, exist_ok=True)
for rel in todo:
    n = os.path.basename(rel)
    local, dst = f"{TMP}/{n}", f"{DRIVE_DIR}/{n}"
    print(f"=== {n} をDL（aria2・16並列・中断してもセル再実行でレジューム） ===", flush=True)
    r = subprocess.run(["aria2c", "-c", "-x16", "-s16", "--file-allocation=none",
                        "--summary-interval=30", "--console-log-level=warn",
                        "-d", TMP, "-o", n, resolved_url(rel, TOKEN)])
    assert r.returncode == 0 and os.path.getsize(local) == SIZES[rel], f"{n} のDL不完全 — このセルを再実行"
    print("  -> Driveへコピー中（FUSE越しで数分〜十数分）", flush=True)
    if os.path.exists(dst):
        os.remove(dst)
    shutil.copy(local, dst)
    assert os.path.getsize(dst) == SIZES[rel], f"{n} のDriveコピー不完全 — このセルを再実行"
    os.remove(local)  # ローカルは都度消してディスクを使い回す
    print("  OK", n)
drive.flush_and_unmount()  # キャッシュをDriveへ書き切ってから終了（これが済むまでランタイムを消さない）
print("完了。全重みをDriveへ配置済み。このランタイムは削除してよい")


In [ ]:
#@title 1. 設定＋環境チェック（毎セッションここだけ編集。実行すると選択中のGPUを表示）
CHAPTERS = []                # 生成するチャプター。空 = バンドル内の全チャプターを番号順に生成。パイロット運用なら ["ch1"] → 合格後 [] （生成済みは自動スキップ）
EXPECTED_GPU = "A100"        # 例 "A100" / "L4"。ランタイムの選択がそれと違うときにここで止まる（設定変更漏れの検知）
BUNDLE_ZIP_FROM_DRIVE = ""   # 例 "/content/drive/MyDrive/41_okayaman_ltx_bundle.zip"。空ならセル4でブラウザからアップロード
WEIGHTS_DRIVE_DIR = "/content/drive/MyDrive/ltx25_weights"    # Driveを重みキャッシュに使う（DL回避。実行前に必要分をローカルへコピーする）
SAVE_WEIGHTS_TO_DRIVE = True # WEIGHTS_DRIVE_DIR設定時、HFから落とした重みをDriveへ保存する（次回セッションが数分で立ち上がる）
OUT_DRIVE_DIR = "/content/drive/MyDrive/ltx25_outputs/"        # 設定すると各チャプター完了ごとに即Driveへ退避（切断事故に強い）
COMFY_FLAGS = []             # 生成中に CUDA out of memory が出たら ["--lowvram"] にしてセル6から再実行
AUTO_SHUTDOWN = False      # Trueにすると、セル7の全チャプター完了後にDriveへ書き切ってからランタイムを自動削除（課金停止。OUT_DRIVE_DIR設定時のみ機能。パイロットはFalseのまま、放置する本番ランでTrueにする）

# --- 以下は自動判定（編集不要） ---
import shutil, torch, psutil
TRANSFORMER = "ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors"
ENCODER = "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors"
VIDEO_VAE = "ltx-2.5-video-vae-bf16.safetensors"
AUDIO_VAE = "ltx-2.5-audio-vae-bf16.safetensors"
if torch.cuda.is_available():
    NAME, CAP = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 2**30
else:
    # 無料CPUランタイム: 生成はできないが、セル2→3で重みをDL→Driveへ配置する用途（0円）に使える
    NAME, CAP, VRAM = "CPU（重み配置専用モード — セル3まで実行、生成セルは不可）", (8, 0), 0.0
    print("⚠ GPUなし: 重みのDrive配置専用モードとして続行")
if EXPECTED_GPU:
    assert EXPECTED_GPU.lower() in NAME.lower(), (
        f"意図したGPU「{EXPECTED_GPU}」と実際の割当「{NAME}」が違う！\n"
        f"ランタイム → ランタイムのタイプを変更 → {EXPECTED_GPU} を選んで再接続してから、このセルを再実行")
if torch.cuda.is_available():
    assert CAP >= (8, 0), f"{NAME} は生成に使えない（Turing以下）。L4以上のGPUを選ぶ"
RAM = psutil.virtual_memory().total / 2**30
DISK = shutil.disk_usage("/content").free / 2**30

print(f"★ このセッションのGPU: {NAME} (SM {CAP[0]}.{CAP[1]})  VRAM {VRAM:.1f} GiB  RAM {RAM:.1f} GiB  空きディスク {DISK:.1f} GiB")
print(f"★ 使用する重み（int8一式・L4/A100共通・GPU別切替なし）: {TRANSFORMER} / {ENCODER}")
print("★ 設定:", dict(CHAPTERS=CHAPTERS, WEIGHTS_DRIVE_DIR=WEIGHTS_DRIVE_DIR or "(未使用)",
                     OUT_DRIVE_DIR=OUT_DRIVE_DIR or "(未使用)"))

# ディスク見積り: 重み一式 約39GB（int8トランスフォーマ21.5＋Gemma4エンコーダ15.4＋VAE約2）。
# H3と違い全部を常時ローカルに置ける（L4の実測ディスク65GBでも収まる）ため、ユニット入れ替えは無い。
if DISK < 46:
    print(f"⚠ 空き{DISK:.0f}GiBに対し重み一式＋作業分で約46GB必要 — 不要ファイルの削除を検討")
if VRAM and VRAM < 30:
    print("⚠ L4級VRAM（22GiB）での1344x768生成は未実測 — OOMになったらCOMFY_FLAGS=[\"--lowvram\"]でセル6から再実行、"
          "それでも駄目なら解像度を1120x640（同アスペクト比・32の倍数）へ落としてworkflowを作り直す")


In [ ]:
#@title ★ 一括実行（セル1〜7をまとめて実行 — セル1を編集したら、あとはこのセルだけ押せばよい）
# セル1→2→…→7を1つずつ押す手間を無くすランナー。Colab標準の「すべてのセルを実行」は
# セル0（無料CPU専用の重み事前配置）やセル9（アドホック生成）まで走ってしまうので使えない。
# ここではノートブック自身のセルソースを取り出し、指定した番号のセルだけを上から順に実行する。
# 操作が必要なもの（バンドルzipのアップロード）は最初にまとめて済ませるので、以降は無人で流れる。
#@markdown - `FROM_STEP` / `TO_STEP`: 実行するセル番号の範囲（既定 1〜7）。失敗したセルから再開するときは `FROM_STEP` をその番号にする（生成済みチャプターはセル7がスキップするのでやり直しは安い）
FROM_STEP = 1  #@param {type:"integer"}
TO_STEP = 7  #@param {type:"integer"}
#@markdown - `FAIL_SHUTDOWN_SEC`: **失敗して止まったとき**、この秒数のカウントダウン後にランタイムを切断・削除して課金を止める（`0` = 切断しない）。カウントダウン中にこのセルを停止（■）すれば取り消せる。Driveへ退避されていない成果物があるときは切断しない。切断前にComfyUIログの末尾を出力へ残す（`OUT_DRIVE_DIR`設定時はログもDriveへコピー）
FAIL_SHUTDOWN_SEC = 600  #@param {type:"integer"}
#@markdown ---
#@markdown バンドルzipをブラウザからアップロードする場合（`BUNDLE_ZIP_FROM_DRIVE`が空のとき）は**最初に**求められる。以降は操作不要。

RUN_ALL_CELL_MARKER = True  # このセル自身を実行対象から外すための目印（消さない）
import glob as _ra_glob, os as _ra_os, re as _ra_re, shutil as _ra_shutil, time as _ra_time

def _ra_load_steps():
    # ノートブックの現在の内容（フォームの編集も反映済み）を取り出し、
    # 先頭行の「#@title <番号>.」/「# <番号>.」から番号→ソースの対応を作る。
    from google.colab import _message
    res = _message.blocking_request("get_ipynb", timeout_sec=120)
    nb = res["ipynb"] if isinstance(res, dict) and "ipynb" in res else res
    steps = {}
    for c in nb["cells"]:
        if c.get("cell_type") != "code":
            continue
        src = c["source"]
        src = src if isinstance(src, str) else "".join(src)
        if "RUN_ALL_CELL_MARKER" in src:
            continue
        for line in src.split("\n")[:3]:
            m = _ra_re.match(r"\s*(?:#@title|#)\s*(\d)[.．]", line)
            if m:
                steps.setdefault(int(m.group(1)), src)
                break
    return steps

def _ra_premount():
    # Driveのマウントは初回に承認ダイアログが出ることがある = 操作が必要。アップロードと一緒に
    # 先に済ませておけば、セル3以降のDriveアクセスで止まらない。
    if not any(globals().get(k) for k in ("WEIGHTS_DRIVE_DIR", "BUNDLE_ZIP_FROM_DRIVE", "OUT_DRIVE_DIR")):
        return
    if _ra_os.path.isdir("/content/drive/MyDrive"):
        return
    from google.colab import drive as _gd
    print("★ 先にGoogle Driveをマウントする（初回は承認ダイアログが出る）", flush=True)
    _gd.mount("/content/drive")

def _ra_preupload():
    # 操作が必要なアップロードを先に済ませる（重いインストール・重み配置の前）。
    # ここで置いたパスを BUNDLE_ZIP_LOCAL に入れると、セル4がアップロードを求めずそれを使う。
    if 4 not in _RA_TODO or globals().get("BUNDLE_ZIP_FROM_DRIVE"):
        return
    cur = globals().get("BUNDLE_ZIP_LOCAL") or ""
    if cur and _ra_os.path.exists(cur):
        print(f"  バンドルzipは既にローカルにある（セル4はこれを使う）: {cur}", flush=True)
        return
    from google.colab import files
    print("★ 先にバンドルzip（<NN>_<slug>_bundle.zip）をアップロードする — 操作が必要なのはここだけ",
          flush=True)
    up = files.upload()
    zips = [n for n in up if n.lower().endswith(".zip")]
    assert zips, "zipが選ばれていない — バンドルzipを選ぶか、セル1で BUNDLE_ZIP_FROM_DRIVE を設定する"
    globals()["BUNDLE_ZIP_LOCAL"] = _ra_os.path.join(_ra_os.getcwd(), zips[0])
    print(f"  アップロード完了: {BUNDLE_ZIP_LOCAL} "
          f"({_ra_os.path.getsize(BUNDLE_ZIP_LOCAL) / 2**20:.1f} MB)。以降は無人で流れる", flush=True)

def _ra_fail_shutdown():
    # 失敗して止まったあと放置されると、GPUランタイムが繋がっている間ずっとCUを食う。
    # 診断材料（ComfyUIログ末尾）を出力に残してから、猶予後にランタイムを削除して課金を止める。
    outdir = globals().get("OUT_DRIVE_DIR") or ""
    for lg in sorted(_ra_glob.glob("/content/comfyui_*.log")):
        try:
            with open(lg, errors="replace") as fo:
                tail = fo.read()[-4000:]
            print(f"\n--- {lg}（末尾） ---\n{tail}", flush=True)
            if outdir and _ra_os.path.isdir(outdir):
                _ra_shutil.copy(lg, outdir)
                print(f"--- {lg} をDriveへ保存: {outdir}", flush=True)
        except OSError as e:
            print(f"--- {lg} を読めなかった: {e}", flush=True)
    if FAIL_SHUTDOWN_SEC <= 0:
        print("（FAIL_SHUTDOWN_SEC=0 のため自動切断しない。放置すると課金が続くので、"
              "終わったら「ランタイム → ランタイムを接続解除して削除」）", flush=True)
        return
    outs = sorted(_ra_glob.glob("/content/outputs/*.mp4"))
    unsaved = [_ra_os.path.basename(o) for o in outs
               if not (outdir and _ra_os.path.exists(_ra_os.path.join(outdir, _ra_os.path.basename(o))))]
    if unsaved:
        print(f"⚠ 自動切断を中止: Driveに退避されていない成果物がある {unsaved} —"
              " セル8で回収してから手動でランタイムを削除すること", flush=True)
        return
    print(f"★ {FAIL_SHUTDOWN_SEC}秒後にランタイムを切断・削除して課金を止める。"
          "取り消すならこのセルを停止（■）", flush=True)
    left, step = FAIL_SHUTDOWN_SEC, max(15, FAIL_SHUTDOWN_SEC // 10)
    while left > 0:
        print(f"   切断まで {left}秒...", flush=True)
        _ra_time.sleep(min(step, left))
        left -= min(step, left)
    try:
        from google.colab import drive as _gd
        _gd.flush_and_unmount()  # Driveへの書き込み（成果物・ログ）を確実に反映させてから消す
        print("   Driveへ書き切った", flush=True)
    except Exception as e:
        print(f"   ⚠ flush_and_unmount に失敗: {e}", flush=True)
    print("   ランタイムを削除する。以降の出力は表示されない", flush=True)
    from google.colab import runtime
    runtime.unassign()

try:
    _RA_STEPS = _ra_load_steps()
except Exception as _ra_e:
    raise SystemExit(f"この環境ではノートブックのセルを取り出せなかった（{type(_ra_e).__name__}: {_ra_e}）"
                     " — セル1〜7を順に手で実行すること")
# 立ち上げの並列化: セル3（重み配置）は呼ぶとバックグラウンドスレッドでコピー／DLを進めるので、
# 番号順（2→3）ではなく先に始める。33〜54GBのコピーがセル2のpip install・セル4のバンドル展開と
# 並走し、その分だけ全体が短くなる（ComfyUIを起動するセル6より前にセル2が終わっていればよい）。
RUN_ORDER = [1, 3, 4, 2, 5, 6, 7]
_RA_RANGE = list(range(FROM_STEP, TO_STEP + 1))
_RA_TODO = [n for n in RUN_ORDER if n in _RA_RANGE]
_RA_SKIP = [n for n in _RA_RANGE if n not in RUN_ORDER]
if _RA_SKIP:
    print(f"（セル{_RA_SKIP} は一括実行の対象外 — セル8=回収・セル9=単発生成は手で実行する）", flush=True)
_RA_MISSING = [n for n in _RA_TODO if n not in _RA_STEPS]
assert not _RA_MISSING, (f"セル{_RA_MISSING} が見つからない — 各セル先頭の「#@title <番号>.」を"
                         "書き換えると検出できなくなる（このセル自身は番号を持たない）")
if FROM_STEP > 1 and "SERVERS" not in globals():
    print("⚠ セル1をこのセッションで実行していない（設定が未定義）— FROM_STEP=1 から流すこと", flush=True)

_ra_ip = get_ipython()
_ra_t0 = _ra_time.time()
_ra_err = None
print(f"★ 一括実行: セル{' → '.join(map(str, _RA_TODO))}（{len(_RA_TODO)}本）"
      "／番号順でないのは、セル3の重みコピーをセル2のpip installと並走させるため", flush=True)
try:
    if FROM_STEP > 1:  # セル1を再実行しない再開時も、必要な操作は先に済ませる
        _ra_premount()
        _ra_preupload()
    for _ra_n in _RA_TODO:
        print(f"\n{'=' * 78}\n▶ セル{_ra_n} 開始（経過 {(_ra_time.time() - _ra_t0) / 60:.1f}分）\n{'=' * 78}",
              flush=True)
        _ra_res = _ra_ip.run_cell(_RA_STEPS[_ra_n])
        if not _ra_res.success:
            raise RuntimeError(f"セル{_ra_n} で失敗（原因は直前のトレースバック）。直したら "
                               f"FROM_STEP={_ra_n} にしてこのセルを再実行すれば続きから流せる")
        print(f"✔ セル{_ra_n} 完了（経過 {(_ra_time.time() - _ra_t0) / 60:.1f}分）", flush=True)
        if _ra_n == 1:
            # 設定が読めた直後 = 重いセル2〜4の前に、操作が必要なものをまとめて済ませる
            _ra_premount()
            _ra_preupload()
except KeyboardInterrupt:
    print("\n■ 停止された（自動切断はしない）。課金を止めるなら「ランタイム → 接続解除して削除」", flush=True)
    raise
except BaseException as _ra_e2:
    _ra_err = _ra_e2

if _ra_err is None:
    print(f"\n★ セル{FROM_STEP}〜{TO_STEP} 完了（合計 {(_ra_time.time() - _ra_t0) / 60:.1f}分）"
          " — 成果物の回収はセル8、単発生成はセル9", flush=True)
else:
    print(f"\n■ 一括実行を中断: {_ra_err}", flush=True)
    _ra_fail_shutdown()
    raise SystemExit(str(_ra_err))

In [ ]:
%%bash
# 2. ComfyUIインストール＋aria2導入（2〜3分）
set -e
# aria2はセル3（重み配置）が先に走るケースで自前導入するため、既に在れば触らない
# （同時にapt-getを叩くとdpkgロックで両方が失敗しうる）
command -v aria2c > /dev/null 2>&1 || apt-get -yq install aria2 > /dev/null 2>&1 || true
cd /content
# 一括実行では重み配置（セル3）・バンドル投入（セル4）を先に走らせて pip install と並走させるため、
# /content/ComfyUI が「models/ や input/ だけ先に作られた状態」で来ることがある。ディレクトリの
# 有無で判定すると clone が丸ごとスキップされてしまうので、main.py の有無で判定し、別の場所へ
# cloneしてから既存ファイルを上書きせずマージする（cp -n）。
if [ ! -f ComfyUI/main.py ]; then
  rm -rf /content/ComfyUI_clone
  git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI_clone
  mkdir -p /content/ComfyUI
  cp -rn /content/ComfyUI_clone/. /content/ComfyUI/
  rm -rf /content/ComfyUI_clone
fi
cd ComfyUI
pip install -q -r requirements.txt
test -f comfy_extras/nodes_lt.py && echo "LTX nodes: OK" \
  || { echo "ERROR: nodes_lt.py が無い — ComfyUIが古い"; exit 1; }
test -f comfy/text_encoders/gemma4.py && echo "Gemma4 encoder (LTX-2.5): OK" \
  || { echo "ERROR: gemma4.py が無い — ComfyUIが古い（LTX-2.5未対応）"; exit 1; }
grep -q "LTXVConcatAVLatent" comfy_extras/nodes_lt.py && echo "LTX AV nodes: OK" \
  || { echo "ERROR: LTXVConcatAVLatentが無い — ComfyUIが古い（LTX-2.x未対応）"; exit 1; }


In [ ]:
#@title 3. 重み配置（バックグラウンド実行 — 開始したらそのままセル4〜6へ進んでよい。セル7の冒頭が完了を待つ）
# LTX-2.5はint8一式（約39GB）がL4/A100共通。全ファイルを常時ローカルに置けるので、
# H3のようなGPU別バリアント切替・ユニット入れ替えは無い。
# Drive優先（8スレッド並列チャンクコピー・レジューム可）→無い分はHFからaria2でDL
# （ゲート付きリポジトリのためHFトークン必要）→Driveへ保存。
import concurrent.futures, glob, os, shutil, subprocess, threading, time, urllib.error, urllib.request
REPO = "https://huggingface.co/Lightricks/LTX-2.5/resolve/main"
SUB = lambda n: ("vae" if "-vae-" in n else
                 "text_encoders" if n.startswith("gemma") else
                 "latent_upscale_models" if "upscaler" in n else "diffusion_models")
SIZES = {  # HF上の正確なバイト数（2026-08時点）。完全性チェックに使う
    "ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors": 21_504_034_224,
    "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors": 15_372_971_786,
    "ltx-2.5-video-vae-bf16.safetensors": 1_472_223_346,
    "ltx-2.5-audio-vae-bf16.safetensors": 364_866_540,
    "ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors": 995_778_752,
    "ltx-2.5-latent-temporal-upscaler-x2-bf16-1.0.safetensors": 261_944_000,
}
need = [TRANSFORMER, ENCODER, VIDEO_VAE, AUDIO_VAE]

if WEIGHTS_DRIVE_DIR or BUNDLE_ZIP_FROM_DRIVE or OUT_DRIVE_DIR:
    from google.colab import drive as _gd
    _gd.mount("/content/drive")  # OAuthが出ることがあるのでここだけ前面で実行
drive_dir = WEIGHTS_DRIVE_DIR or None
if drive_dir:
    os.makedirs(drive_dir, exist_ok=True)

def ok_size(path, n):
    return os.path.exists(path) and os.path.getsize(path) == SIZES[n]

def hf_token():
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            return t
    except Exception:
        pass
    import getpass
    return getpass.getpass("HFトークン（read・シークレット未登録のため手入力）: ")

def resolved_url(rel, token):
    # ゲート付きリポジトリ: トークン付きでリダイレクトを解決し、署名済みCDN URLを得てから
    # aria2はヘッダ無しでDLする（AuthorizationヘッダをCDNへ転送すると拒否されることがあるため）。
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, *a, **k):
            return None
    req = urllib.request.Request(f"{REPO}/{rel}", method="HEAD",
                                 headers={"Authorization": f"Bearer {token}"})
    try:
        urllib.request.build_opener(NoRedirect).open(req, timeout=60)
        return req.full_url
    except urllib.error.HTTPError as e:
        if e.code in (301, 302, 303, 307, 308):
            return e.headers["Location"]
        if e.code in (401, 403):
            raise RuntimeError(
                "HFのライセンス未承諾かトークン権限不足 — https://huggingface.co/Lightricks/LTX-2.5 "
                "で Agree and Access を押し、READ権限（gated reposアクセス可）のトークンを使う")
        raise

# DLが必要になるかを前面で判定し、必要ならトークンもここ（前面）で確保する
# （バックグラウンドスレッド内のgetpassは入力できないため）
TOKEN = None
if any(not ok_size(f"/content/ComfyUI/models/{SUB(n)}/{n}", n)
       and not (drive_dir and ok_size(f"{drive_dir}/{n}", n)) for n in need):
    TOKEN = hf_token()

DRIVE_IO_LOCK = threading.Lock()  # キャッシュ掃除のunmountと、他セルのDrive読み出しの衝突防止

# --- ディスク管理ヘルパ（逼迫時の自動掃除と、不足時に原因が分かる診断） ---
GiB = 2**30
COPY_FLOOR = 2 * GiB   # 大物コピー中に残しておきたい空き（DriveFSの読み出しキャッシュ用ヘッドルーム）
DIFF_DIR = "/content/ComfyUI/models/diffusion_models"
DRIVEFS_CACHE_DIRS = ["/root/.config/Google/DriveFS", "/root/.cache/Google/DriveFS"]

def headroom_for(remaining):
    # コピーに必要な空き = 残りバイト ＋ ヘッドルーム。ヘッドルームは残りに応じて縮める
    # （残り1GiBのレジューム時に「4GiB空いていないから不足」と落ちるのを防ぐ）。
    return remaining + min(COPY_FLOOR, max(remaining // 4, 512 * 2**20))

def free_bytes():
    return shutil.disk_usage("/content").free

def path_bytes(p):
    if os.path.isfile(p) and not os.path.islink(p):
        return os.path.getsize(p)
    total = 0
    for root, _d, fs in os.walk(p, onerror=lambda e: None):
        for f in fs:
            try:
                total += os.lstat(os.path.join(root, f)).st_size
            except OSError:
                pass
    return total

def deleted_open_bytes():
    # 削除済みなのにプロセスが開いたままのファイルは、dfの空きとして戻ってこない（ComfyUIが
    # unetをmmapしたまま等）。「消したのに空きが増えない」ときの原因を特定するために合計を見る。
    total, seen = 0, set()
    for fd in glob.glob("/proc/[0-9]*/fd/*"):
        try:
            if not os.readlink(fd).endswith(" (deleted)"):
                continue
            st = os.stat(fd)
        except OSError:
            continue
        if (st.st_dev, st.st_ino) in seen:
            continue
        seen.add((st.st_dev, st.st_ino))
        total += st.st_size
    return total

def biggest_files(n=6, roots=("/content", "/root"), floor=512 * 2**20):
    hits = []
    for r in roots:
        for root, dirs, fs in os.walk(r, onerror=lambda e: None):
            dirs[:] = [d for d in dirs if os.path.join(root, d) != "/content/drive"]  # Drive側は数えない
            for f in fs:
                p = os.path.join(root, f)
                try:
                    sz = os.lstat(p).st_size
                except OSError:
                    continue
                if sz >= floor:
                    hits.append((sz, p))
    return sorted(hits, reverse=True)[:n]

def disk_short_msg(need, remaining=None):
    # 「空きは残っているのにエラー」を避けるため、判断に使った数字をそのまま出す。
    du = shutil.disk_usage("/content")
    msg = (f"ディスク不足: 空き {du.free / GiB:.1f} GiB / 全体 {du.total / GiB:.1f} GiB、"
           f"必要 {need / GiB:.1f} GiB")
    if remaining is not None:
        msg += f"（コピー残り {remaining / GiB:.1f} GiB ＋ 作業用ヘッドルーム）"
    dob = deleted_open_bytes()
    if dob >= GiB:
        msg += (f"\n  → 削除済みなのにプロセスが掴んでいるファイルが {dob / GiB:.1f} GiB ある"
                "（ComfyUIがunetをmmapしたまま等）。セル6を再実行してComfyUIを再起動すれば解放される")
    big = biggest_files()
    if big:
        msg += "\n  → ローカルの大きいファイル: " + ", ".join(f"{p}={s / GiB:.1f}GiB" for s, p in big)
    return msg + "\n  → 不要ファイルを整理してこのセルを再実行（コピーは途中から再開する）"

def purge_drivefs_cache():
    # DriveFSはFUSE読み出しの内容キャッシュをローカルディスクにも書くため、大物コピー中に
    # 「コピー先＋キャッシュ」の二重消費でディスクが枯渇することがある（ENOSPC実測・2026-08）。
    # unmount→キャッシュ削除→remountで空ける。掃除で実際に何GiB空いたかを必ず表示する
    # （0GiBなら原因はキャッシュではない＝重み本体か、プロセスが掴んだ削除済みファイル）。
    from google.colab import drive as _gd
    lock = globals().get("DRIVE_IO_LOCK")
    if lock:
        lock.acquire()
    try:
        before = free_bytes()
        hit = [d for d in DRIVEFS_CACHE_DIRS if os.path.isdir(d)]
        print(f"  空きディスク逼迫（空き {before / GiB:.1f} GiB）→ DriveFSキャッシュを掃除"
              f"（unmount→削除→remount・数十秒）: {', '.join(hit) or '(キャッシュ無し)'}", flush=True)
        _gd.flush_and_unmount()
        for d in hit:
            shutil.rmtree(d, ignore_errors=True)
        _gd.mount("/content/drive")
        after = free_bytes()
        print(f"  キャッシュ掃除で {max(0, after - before) / GiB:.1f} GiB 解放"
              f"（空き {after / GiB:.1f} GiB）", flush=True)
        return after
    finally:
        if lock:
            lock.release()

def reclaim_disk(need, keep=()):
    # needバイトの空きを作る。安いものから順に捨て、都度measureし直して足りたら止める
    # （1手が効かなかったときに次の手へ進めるようにする）。戻り値は最終的な空きバイト。
    free = free_bytes()
    if free >= need:
        return free
    # 1) 他のunetのローカル実体（Driveに実体があるので消してよい。symlinkは実体を持たないので対象外）
    for other in sorted(glob.glob(f"{DIFF_DIR}/*.safetensors") + glob.glob(f"{DIFF_DIR}/*.part")):
        if free >= need:
            return free
        if other in keep or os.path.islink(other):
            continue
        sz = path_bytes(other)
        os.remove(other)
        print(f"  ディスク確保のためローカルunetを削除（Driveに実体あり）: "
              f"{os.path.basename(other)} -{sz / GiB:.1f} GiB", flush=True)
        free = free_bytes()
    # 2) 生成に不要なキャッシュ・作業ファイル
    junk_list = ["/root/.cache/pip", "/root/.cache/huggingface", "/root/.cache/torch",
                 "/content/h3_dl", "/content/ComfyUI/temp"]
    if glob.glob("/content/bundle/**/script.md", recursive=True):  # 展開済みならバンドルzipは不要
        junk_list += [z for z in sorted(glob.glob("/content/*.zip"))
                      if os.path.basename(z) != "h3_outputs.zip"]  # 回収用zipは消さない
    for junk in junk_list:
        if free >= need:
            return free
        if junk in keep or not os.path.exists(junk):
            continue
        sz = path_bytes(junk)
        if sz < 200 * 2**20:
            continue
        if os.path.isdir(junk) and not os.path.islink(junk):
            shutil.rmtree(junk, ignore_errors=True)
        else:
            os.remove(junk)
        print(f"  ディスク確保のため削除: {junk} -{sz / GiB:.1f} GiB", flush=True)
        free = free_bytes()
    # 3) DriveFSの読み出しキャッシュ（remountに数十秒かかるので最後）
    if free < need:
        free = purge_drivefs_cache()
    return free

def copy_from_drive(src, dst, threads=8, chunk=64 * 2**20):
    # Drive→ローカルのレジューム可能コピー。並列pread（8スレッド・実測62→83MB/s）で読み、
    # 追記順を守って書く＝dstのファイルサイズがそのままレジューム点になる。
    # 空きが逼迫したらキャッシュ掃除等で空けて続行する（shutil.copyだとENOSPCで落ちる）。
    size = os.path.getsize(src)
    done = os.path.getsize(dst) if os.path.exists(dst) else 0

    def read_round(start):  # startからthreads*chunk分を並列preadで読んで返す
        n = min(threads * chunk, size - start)
        def one(i):
            off, ln = start + i * chunk, min(chunk, n - i * chunk)
            if ln <= 0:
                return b""
            fd = os.open(src, os.O_RDONLY)
            try:
                parts, got = [], 0
                while got < ln:
                    b = os.pread(fd, ln - got, off + got)
                    assert b, f"{src} の読み出しが途切れた — セルの再実行で続きから再開する"
                    parts.append(b)
                    got += len(b)
                return b"".join(parts)
            finally:
                os.close(fd)
        with concurrent.futures.ThreadPoolExecutor(threads) as ex:
            return b"".join(ex.map(one, range(threads)))

    with concurrent.futures.ThreadPoolExecutor(1) as ahead:
        nxt = None  # 先読み: 書き込みと次ラウンドの読みを重ねる
        while done < size:
            buf = nxt.result() if nxt else None
            nxt = None
            # 必要な空きは「残りのバイト＋ヘッドルーム」。残りが少ないときにヘッドルームを
            # 理由に止めない（空きが残っているのに落ちるのを防ぐ）。
            want = headroom_for(size - done)
            if free_bytes() < want:
                free = reclaim_disk(want, keep=(dst,))
                assert free >= want, disk_short_msg(want, size - done)
            if buf is None:
                buf = read_round(done)
            if done + len(buf) < size:
                nxt = ahead.submit(read_round, done + len(buf))
            with open(dst, "ab") as fo:
                fo.write(buf)
            if done // (4 * 2**30) != (done + len(buf)) // (4 * 2**30):
                print(f"    ... {(done + len(buf)) / 2**30:.0f}/{size / 2**30:.0f} GiB", flush=True)
            done += len(buf)

def ensure_aria2(timeout=240):
    """aria2cを使える状態にする。無ければapt-getで入れる（十数秒）。

    ★一括実行はセル3（重み配置）をセル2（ComfyUIインストール＋aria2導入）より先に流す
    ＝aria2のapt-getがまだ走っていない状態でここに来る（H3ノートブックで実測:
    FileNotFoundError: 'aria2c' で重み配置が丸ごと失敗した）。セル2のapt-getと同時に
    走るとdpkgロックで失敗するので、取れるまで少し待って再試行する。"""
    if shutil.which("aria2c"):
        return True
    deadline = time.time() + timeout
    while True:
        r = subprocess.run(["apt-get", "-yq", "install", "aria2"],
                           capture_output=True, text=True)
        if shutil.which("aria2c"):
            print("★ aria2 を導入（セル2のapt-getを待たずにここで入れた）", flush=True)
            return True
        if time.time() >= deadline:
            tail = ((r.stderr or r.stdout).strip().splitlines() or ["(出力なし)"])[-1]
            print(f"⚠ aria2 の導入に失敗（{timeout}秒リトライした）: {tail}", flush=True)
            return False
        time.sleep(5)  # 大半はセル2のapt-getとのdpkgロック競合 = 数秒待てば取れる


def _place_weights():
    for n in need:
        dst = f"/content/ComfyUI/models/{SUB(n)}/{n}"
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if ok_size(dst, n):
            print("skip（配置済み）", n)
            continue
        drv = f"{drive_dir}/{n}" if drive_dir else None
        if drv and ok_size(drv, n):
            # Drive(FUSE)越しのsymlink直参照は実行時に使えない（H3で実測）ため、必ずローカルへコピーする
            print(f"Driveからローカル化中（8スレッド並列コピー。中断してもレジューム可）: {n}", flush=True)
            copy_from_drive(drv, dst + ".part")
            os.replace(dst + ".part", dst)
            print("drive OK（ローカル化済み）", n)
            continue
        assert TOKEN, f"{n} のDLにHFトークンが必要 — セル3を再実行してトークンを入力"
        assert ensure_aria2(), "aria2 を導入できない — セル2を実行してからこのセルを再実行"
        print(f"=== {n} をaria2でDL（16並列・15秒毎に進捗表示・中断してもレジューム可） ===", flush=True)
        r = subprocess.run(["aria2c", "-c", "-x16", "-s16", "--file-allocation=none",
                            "--summary-interval=15", "--console-log-level=warn",
                            "-d", os.path.dirname(dst), "-o", n,
                            resolved_url(f"{SUB(n)}/{n}", TOKEN)])
        assert r.returncode == 0 and ok_size(dst, n), f"{n} のDLに失敗 — このセルを再実行すれば途中から再開する"
        print("hf OK", n)
        if drv and SAVE_WEIGHTS_TO_DRIVE:
            try:
                print(f"  -> Driveへ保存中（FUSE越しで時間がかかる。次回以降の高速化用）: {drv}", flush=True)
                shutil.copy(dst, drv)
            except OSError as e:  # Drive容量・一時キャッシュ枯渇などでも生成は止めない
                print(f"  ⚠ Drive保存に失敗（生成には影響なし。後で無料CPUセッションでの配置を推奨）: {e}")
                if os.path.exists(drv):
                    os.remove(drv)
    print(f"★ 重み配置 完了。空きディスク: {shutil.disk_usage('/content').free / 2**30:.1f} GiB", flush=True)

# 配置はバックグラウンドで実行し、バンドル投入（セル4）〜ComfyUI起動（セル6）と並行させる。完了はセル7の冒頭で待つ。
if "WEIGHTS_THREAD" in globals() and WEIGHTS_THREAD.is_alive():
    print("前回の重み配置がまだ実行中 — 完了を待ってから続行")
    WEIGHTS_THREAD.join()
WEIGHTS_ERR = []

def _bg_place():
    try:
        _place_weights()
    except BaseException as e:
        WEIGHTS_ERR.append(e)
        print(f"⚠ 重み配置が失敗: {e} — セル3を再実行（コピー/DLは途中から再開する）", flush=True)

def wait_weights():
    if WEIGHTS_THREAD.is_alive():
        print("重み配置（バックグラウンド）の完了を待機中...", flush=True)
    WEIGHTS_THREAD.join()
    assert not WEIGHTS_ERR, f"重み配置が失敗している: {WEIGHTS_ERR[0]} — セル3を再実行"

WEIGHTS_THREAD = threading.Thread(target=_bg_place, daemon=True)
WEIGHTS_THREAD.start()
print("★ 重み配置をバックグラウンドで開始 — このままセル4〜6を進めてOK（セル7の冒頭で完了を待つ）")


In [ ]:
#@title 4. バンドル投入（zipをアップロード or Driveから）→ ComfyUI/input/ へ配備
import glob, os, shutil, threading, zipfile

if BUNDLE_ZIP_FROM_DRIVE:
    assert os.path.exists(BUNDLE_ZIP_FROM_DRIVE), f"{BUNDLE_ZIP_FROM_DRIVE} が無い"
    zp = "/content/" + os.path.basename(BUNDLE_ZIP_FROM_DRIVE)
    # セル3のバックグラウンド重み配置がDriveFSキャッシュ掃除（unmount）をすることがあるため、
    # Driveからの読み出しはロックを取ってローカルへ写してから使う
    with globals().get("DRIVE_IO_LOCK") or threading.Lock():
        shutil.copy(BUNDLE_ZIP_FROM_DRIVE, zp)
elif globals().get("BUNDLE_ZIP_LOCAL") and os.path.exists(BUNDLE_ZIP_LOCAL):
    zp = BUNDLE_ZIP_LOCAL  # ★一括実行セルが最初にアップロードしておいたzip（以降を無人で流すため）
    print("アップロード済みのzipを使う:", zp)
else:
    from google.colab import files
    print("バンドルzip（<NN>_<slug>_bundle.zip）を選択:")
    up = files.upload()
    zp = os.path.join(os.getcwd(), next(iter(up)))

shutil.rmtree("/content/bundle", ignore_errors=True)
zipfile.ZipFile(zp).extractall("/content/bundle")
hits = glob.glob("/content/bundle/**/script.md", recursive=True)
assert hits, "zip内にscript.mdが見つからない — ラン専用ディレクトリごとzipしたか確認"
BUNDLE = os.path.dirname(hits[0])
print("BUNDLE =", BUNDLE)

inp = "/content/ComfyUI/input"
os.makedirs(inp, exist_ok=True)
n = 0
for p in sorted(glob.glob(f"{BUNDLE}/*.png") + glob.glob(f"{BUNDLE}/*.wav")):
    if os.path.basename(p).startswith("ref_canvas_"):
        continue
    shutil.copy(p, inp)
    n += 1
print(f"{n} files -> ComfyUI/input/")


In [ ]:
#@title 5. workflowの検証（重み名・8k+1グリッド・SaveVideoのcodec補完）
import glob, json, os
KNOWN = {TRANSFORMER, ENCODER, VIDEO_VAE, AUDIO_VAE}
wfs = sorted(glob.glob(f"{BUNDLE}/ch*_workflow.json"))
assert wfs, f"{BUNDLE} に ch*_workflow.json が無い — build_ltx25_workflow.py で生成したか確認"
for wf in wfs:
    with open(wf) as f:
        d = json.load(f)
    frames = width = height = None
    for node in d.values():
        ins = node.get("inputs", {})
        if node.get("class_type") == "EmptyLTXVLatentVideo":
            frames, width, height = ins.get("length"), ins.get("width"), ins.get("height")
        if node.get("class_type") == "SaveVideo":  # ComfyUI新版(2026-08〜)はcodec必須
            ins.setdefault("codec", "auto")
            ins.setdefault("format", "auto")
        for v in ins.values():
            if isinstance(v, str) and v.endswith(".safetensors") and v not in KNOWN:
                print(f"⚠ {os.path.basename(wf)}: 重み {v} はこのノートブックの配置対象外 — 意図した指定か確認")
    assert frames and (frames - 1) % 8 == 0, \
        f"{os.path.basename(wf)}: framesが8k+1グリッドに乗っていない（{frames}）— H3の17k+5と取り違えていないか確認"
    with open(wf, "w") as f:
        json.dump(d, f, indent=1)
    print(f"OK {os.path.basename(wf)}: {width}x{height} {frames}f ({frames/24:.2f}s @24fps)")


In [ ]:
#@title 6. ComfyUI起動（プロセスが落ちたらこのセルを再実行）
import json, subprocess, sys, time, urllib.request
SERVER = "127.0.0.1:8188"
subprocess.run(["pkill", "-f", "main.py --listen"], check=False)
time.sleep(2)
LOG = open("/content/comfyui.log", "w")
PROC = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", *COMFY_FLAGS],
    cwd="/content/ComfyUI", stdout=LOG, stderr=subprocess.STDOUT)
INFO = None
for _ in range(90):
    try:
        INFO = json.load(urllib.request.urlopen(f"http://{SERVER}/object_info", timeout=5))
        break
    except Exception:
        time.sleep(2)
assert INFO, "ComfyUIが起動しない — !tail -50 /content/comfyui.log で確認"
need_nodes = ["LTXVAddGuide", "LTXVConcatAVLatent", "LTXVEmptyLatentAudio",
              "LTXVSeparateAVLatent", "LTXVCropGuides", "ManualSigmas"]
missing = [k for k in need_nodes if k not in INFO]
assert not missing, f"LTXノードが足りない: {missing} — ComfyUIのバージョンを確認"
print("LTX-2.5 nodes: OK", need_nodes)


In [ ]:
#@title 7. チャプター生成（生成済みはスキップ＝中断・再開に強い。完了ごとに即Drive退避）
import glob, json, os, re, shutil, time, urllib.error, urllib.parse, urllib.request, uuid
if "wait_weights" in globals():
    wait_weights()  # セル3のバックグラウンド重み配置の完了を待つ
if not CHAPTERS:  # 未指定 = バンドル内の全チャプター
    CHAPTERS = sorted(
        (os.path.basename(w)[: -len("_workflow.json")] for w in glob.glob(f"{BUNDLE}/ch*_workflow.json")),
        key=lambda ch: int(re.sub(r"\D", "", ch) or 0))
    print("CHAPTERS未指定 → 全チャプターを生成")
assert CHAPTERS, f"{BUNDLE} に ch*_workflow.json が無い — バンドルを確認"
print("生成順:", CHAPTERS)
os.makedirs("/content/outputs", exist_ok=True)
if OUT_DRIVE_DIR:
    os.makedirs(OUT_DRIVE_DIR, exist_ok=True)

def run_workflow(wf_path, out_path):
    # h3_run.py相当のノートブック内蔵ランナー: workflowを投入→完了をポーリング→動画を回収
    with open(wf_path) as f:
        graph = json.load(f)
    payload = json.dumps({"prompt": graph, "client_id": uuid.uuid4().hex}).encode()
    req = urllib.request.Request(f"http://{SERVER}/prompt", data=payload,
                                 headers={"Content-Type": "application/json"})
    try:
        queued = json.load(urllib.request.urlopen(req, timeout=60))
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"ComfyUIがworkflowを拒否: {e.read().decode('utf-8', 'replace')[:2000]}")
    pid = queued.get("prompt_id")
    assert pid, f"prompt_idが無い: {queued}"
    started = time.time()
    while True:
        time.sleep(15)
        hist = json.load(urllib.request.urlopen(f"http://{SERVER}/history/{pid}", timeout=60))
        entry = hist.get(pid)
        if entry:
            st = entry.get("status", {})
            if st.get("status_str") == "error":
                raise RuntimeError(
                    f"生成失敗: {json.dumps(st.get('messages', []), ensure_ascii=False)[:2000]}")
            if st.get("completed") or entry.get("outputs"):
                break
        e = int(time.time() - started)
        print(f"waiting... {e // 60}m{e % 60:02d}s", flush=True)
    for node_output in entry.get("outputs", {}).values():
        for key in ("videos", "gifs", "images"):
            for item in node_output.get(key, []):
                if item.get("filename", "").lower().endswith((".mp4", ".webm", ".mov")):
                    q = urllib.parse.urlencode({"filename": item["filename"],
                                                "subfolder": item.get("subfolder", ""),
                                                "type": item.get("type", "output")})
                    with urllib.request.urlopen(f"http://{SERVER}/view?{q}", timeout=600) as r, \
                         open(out_path, "wb") as fo:
                        shutil.copyfileobj(r, fo)
                    e = int(time.time() - started)
                    print(f"done in {e // 60}m{e % 60:02d}s -> {out_path}")
                    return
    raise RuntimeError(f"出力に動画ファイルが無い: {json.dumps(entry.get('outputs', {}), ensure_ascii=False)[:2000]}")

for ch in CHAPTERS:
    wf = os.path.join(BUNDLE, f"{ch}_workflow.json")
    assert os.path.exists(wf), f"{wf} が無い"
    out = f"/content/outputs/{ch}.mp4"
    if os.path.exists(out):
        print("skip（生成済み）", ch)
        continue
    print(f"=== {ch} 生成開始（蒸留8ステップ。所要時間は未実測 — パイロットで計測し以後の目安にする）===", flush=True)
    run_workflow(wf, out)
    if OUT_DRIVE_DIR:
        with globals().get("DRIVE_IO_LOCK") or __import__("threading").Lock():
            shutil.copy(out, OUT_DRIVE_DIR)
        print(f"  -> Drive退避済み: {OUT_DRIVE_DIR}/{ch}.mp4")
print("指定チャプター完了")
if AUTO_SHUTDOWN and OUT_DRIVE_DIR:
    # 生成失敗時はここに来ない（例外で停止）: ログ診断のためセッションは残る
    print("Driveへ書き切ってからランタイムを削除する（課金停止）...", flush=True)
    from google.colab import drive as _gd, runtime as _rt
    with globals().get("DRIVE_IO_LOCK") or __import__("threading").Lock():
        _gd.flush_and_unmount()  # DriveFSキャッシュを書き切る（これを飛ばすと最後のmp4がDriveに残らないことがある）
    _rt.unassign()  # 「ランタイム → ランタイムを接続解除して削除」と同じ。以降のセルは実行できない
elif AUTO_SHUTDOWN:
    print("⚠ OUT_DRIVE_DIRが未設定のため自動切断を中止（切断するとローカルのmp4が消える）— セル8で回収してから手動で削除")
else:
    print("接続は継続中（課金も継続）。ブラウザにも落とすならセル8へ。済んだら手動で「ランタイム → ランタイムを接続解除して削除」"
          "（次回からはセル1の AUTO_SHUTDOWN = True で自動化できる）")


In [ ]:
#@title 8. 成果物の回収（zip→ブラウザDL。Drive退避済みならスキップ可）
import glob, subprocess
outs = sorted(glob.glob("/content/outputs/*.mp4"))
assert outs, "/content/outputs にmp4が無い"
print(*outs, sep="\n")
subprocess.run(["zip", "-j", "-q", "/content/ltx25_outputs.zip", *outs], check=True)
from google.colab import files
files.download("/content/ltx25_outputs.zip")
print("回収したら「ランタイム → ランタイムを接続解除して削除」で課金を止めること")


In [ ]:
#@title 9.（任意）アドホック生成 — キーフレーム・プロンプトを直接指定して1本作る
# チャプター定義に縛られない単発生成（開始/終了フレーム条件付けのI2V）。素材はバンドル同梱ファイル名で
# 指定（新素材は左のファイルペインで /content/ComfyUI/input/ へドラッグ＆ドロップしてから指定）。
# framesは8k+1グリッド（97, 121, 145, 193, ...）@24fps。セリフ（リップシンク）は不可 — H3ノートブックを使う。
ADHOC = dict(
    frames=121,
    width=1344, height=768,   # 32の倍数。OOMなら1120x640（同アスペクト比）
    seed=42,
    prompt="A cinematic shot of ... The video starts EXACTLY on the attached first frame and ends "
           "EXACTLY on the attached last frame. ... Soundscape: ... Music: no background music.",
    first="chN_start.png",
    last="chN_end.png",       # "" にすると開始フレームのみの条件付け
    out="adhoc1",
)
import os, subprocess, sys
builder = os.path.join(BUNDLE, "build_ltx25_workflow.py")
assert os.path.exists(builder), \
    "バンドルに build_ltx25_workflow.py が無い — スキルディレクトリからコピーしてzipし直す"
pf = os.path.join(BUNDLE, f"{ADHOC['out']}_prompt.txt")
with open(pf, "w") as f:
    f.write(ADHOC["prompt"])
wf = os.path.join(BUNDLE, f"{ADHOC['out']}_workflow.json")
cmd = [sys.executable, builder, "--out", wf, "--prompt-file", pf,
       "--frames", str(ADHOC["frames"]), "--width", str(ADHOC["width"]),
       "--height", str(ADHOC["height"]), "--seed", str(ADHOC["seed"]),
       "--prefix", f"video/{ADHOC['out']}", "--first", ADHOC["first"]]
if ADHOC["last"]:
    cmd += ["--last", ADHOC["last"]]
subprocess.run(cmd, check=True)
if "wait_weights" in globals():
    wait_weights()  # セル3のバックグラウンド重み配置の完了を待つ
os.makedirs("/content/outputs", exist_ok=True)
run_workflow(wf, f"/content/outputs/{ADHOC['out']}.mp4")  # ランナーはセル7で定義
print("完了 -> セル8で回収")


## 後工程（ローカル）

回収した`chN.mp4`をラン専用ディレクトリに置き、H3分のチャプターと合わせてffmpegでconcat結合する（local-video SKILL.mdのステップ8〜9と同じ）。

音声の扱い: LTX-2.5の音声はプロンプトのSoundscape/Music記述からの**自動生成**（H3のようなwav同梱ではない）。パイロットで環境音を確認し、合わなければローカルでffmpegの`-an`＋差し替えにフォールバックする。セリフ入りチャプターはこのノートブックでは作らない（H3側で生成）。

トラブル時はセル出力と `!tail -80 /content/comfyui.log` をClaude Code / Cursorに貼れば診断できる。
